In [1]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
api_key = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

In [3]:
# Setup LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.7,
)

In [12]:
# 2. Define a simple dummy tool for the agent
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

In [14]:
# 3. Create the Agent with PII Middleware (The Guardrails!)
agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        # Layer 1: Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Layer 2: Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Layer 3: Block API keys entirely - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully using Gemini! 🚀")

Agent with PII middleware created successfully using Gemini! 🚀


In [16]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is ram.kumar@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

content = result["messages"][-1].content

print(" Clean Gemini Response ")
if isinstance(content, list):
    print(content[0]['text'])
else:
    print(content)

d:\Ai Engineer_\LangGraphLearn\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\Ai Engineer_\LangGraphLearn\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


 Clean Gemini Response 
I have located your account associated with **[REDACTED_EMAIL]** and card ending in **5100**. 

How can I assist you today? Please let me know what you need help with (e.g., checking recent charges, updating account details, managing a subscription, or checking an order status).


# DAY_2  Human-in-the-Loop (HITL) & Custom Guardrails

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

# 1. Define Tools
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# 2. Create agent with HITL middleware
hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence[cite: 1]
)
print("Human-in-the-Loop agent created!")

# 3. Execution Phase - Agent pauses
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)
print(" Agent paused — awaiting human approval")
# 4. Human Approval Phase - Agent resumes
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session[cite: 1]
)
print("Approved! Final response")
print(approved_result["messages"][-1].content)

Human-in-the-Loop agent created!


d:\Ai Engineer_\LangGraphLearn\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


 Agent paused — awaiting human approval


d:\Ai Engineer_\LangGraphLearn\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Approved! Final response
[{'type': 'text', 'text': 'The email regarding the Q4 results has been successfully sent to team@company.com.', 'extras': {'signature': 'EqsBCqgBARFNMg/MBNkZOPx68dXMp3CKdtmdijMbH8XGsogZqgq2TLD9+bANLGciLqajRlGjkH+D6bigC7B4mhUY1NKI95xPct+UtFM8ienyYntcb3rQrxH/trXoDU/mAhnjtoxMTOtqcWBOBHpz4HhkBLjk1p/lWeaW03e3R8IPwX0HvxzFQRBO8veajKb4WtYaC+sqEXyyxYvWTQIL01rPPkmC3Jtk54WIHMtT'}}]


In [8]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime

# 1. Define the Custom Middleware Class
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.[cite: 1]
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.[cite: 1]
    """
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
                    }],
                    "jump_to": "end"
                }
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# 2. Create the Agent
filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

# 3. Test the Agent
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


In [ ]:
from langchain_core.messages import AIMessage

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """
    def __init__(self):
        super().__init__()
        self.safety_model = llm # Using Gemini as the judge

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        raw_content = result.content
        
        if isinstance(raw_content, list):
            raw_content = raw_content[0].get('text', '')
        
       
        if "UNSAFE" in str(raw_content).upper():
            print("⚠️ Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

safe_agent = create_agent(
    model=llm,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)


result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make a bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

d:\Ai Engineer_\LangGraphLearn\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\Ai Engineer_\LangGraphLearn\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Response:
[{'type': 'text', 'text': 'I cannot fulfill this request. I am programmed to not provide instructions or information on how to create weapons, explosives, or other dangerous materials.', 'extras': {'signature': 'EqcECqQEARFNMg99xR3aHa53b9iHR5F5eNhtkGp+ReIVyDJCk5X7UGk2ED13LYHmAjtH3T1JrK4M4+lvjZWaKsL3ZHQ5Bldpd05G9VmOKjNMU6+82nmhiwy23kLnIN/VrpJTaIIsx6MM7KYIrbxY7fgEnOS2TqWyVyRXg7ILF65PhULEvFNeZBqRG+GzGmF6SDahqDr4U+WNX+oIEkeDDJQwiVMq6z0iF1kFHpGao6FsKRUCeP6BUbbhHkLvAXpk9dk1dKHdpGVX00OReF76LxC02tUwCt578SkfRu0jabBfhR4pqegB1uYNSg3lo2nWO+W4O2VChnU952iBnAYz61dVltMkt2fuvPZ/iJNHWL3DJ9EXtyKTlDkNJ+XGUJyn9L++CiCBZ9vTVNfk9V2/IJzQRlPhk9Duc6kCNd7B2jcH2Xxw+Q/U6jvn1kUEDyXwY/HS3fo+XZh2jMEQIIUOT9sYL7/2RP0HZ+p4Dw1zPpgZvX85Pn7S2dWjP9h/A6xCCOrH1K8tM43PgYNeUVlufW1mJ6PrqXEZ6MaavoE5trjTLsbxsVHAqv8SVfhgd0DlYJCKlVXOg8tmUXp7tuLM9kxJ7TvcHJUUfF4zXJOLBJHE3iW07D7tELmDcQbV92T5LrxPV0XfDAnBIuCKLZvJelIZSg3TJSo294Q5r+Lw7PLJykdn005KnnjiW5lX+MOa8iS5L9aS9zY0xkei/wDCpe7MsGaI7unIkLM='}}]
